# Lab W1: Tabular Output Heads

Lab ini ditulis tanpa import function. Setiap operasi ditulis eksplisit agar prosesnya terlihat.

**Tujuan:** Memahami bagaimana MLP memproses data tabular - dari input, melalui beberapa layer, sampai ke output dengan tiga jenis tugas berbeda.

## Fase 1: Layer dan Aktivasi

Kita mulai dari `nn.Linear`, layer paling dasar di PyTorch. Contoh di bawah menggunakan satu baris data dengan 10 fitur numerik.

In [ ]:
import torch

# Kita buat 1 baris data dummy dengan 10 fitur secara acak
x = torch.randn(1, 10)

print("Nilai data:", x)
print("Bentuk (shape) data:", x.shape)

`nn.Linear` menerima input berukuran tertentu dan menghasilkan output berukuran lain melalui operasi: `output = input @ W.T + b`.
Mari kita teruskan data ke layer `Linear` yang mengubah 10 fitur menjadi 64 fitur.

In [ ]:
import torch.nn as nn

layer_pertama = nn.Linear(in_features=10, out_features=64)
hasil_layer1 = layer_pertama(x)

print("Shape asal:", x.shape)
print("Shape baru:", hasil_layer1.shape)

Layer linear saja tidak cukup untuk mempelajari pola yang rumit. Kita butuh "aktivasi" untuk memberikan efek non-linear. 
Aktivasi yang paling sering dipakai adalah **ReLU** (Rectified Linear Unit). Aturannya sederhana: jika angkanya negatif, ubah jadi nol. Jika positif, biarkan saja.

In [ ]:
aktivasi = nn.ReLU()
hasil_aktivasi = aktivasi(hasil_layer1)

print("Sebelum ReLU (5 angka pertama):", hasil_layer1[0, :5].detach())
print("Sesudah ReLU (5 angka pertama):", hasil_aktivasi[0, :5].detach())
# Perhatikan bahwa semua angka minus sekarang menjadi 0.

## Fase 2: Membuat Model MLP

Kita rangkai beberapa layer menjadi satu model menggunakan `nn.Sequential`.
Untuk memprediksi **3 kelas** (multikelas), output akhir harus berukuran 3.

In [ ]:
model_multikelas = nn.Sequential(
    nn.Linear(10, 64),  # Layer 1: 10 -> 64
    nn.ReLU(),          # Aktivasi
    nn.Linear(64, 3)    # Output Head: 64 -> 3 (karena kita minta 3 kelas)
)

print(model_multikelas)

**Uji Coba Awal (Smoke Test)**
Sebelum melatih, periksa apakah satu baris data bisa diproses model tanpa error.

In [ ]:
prediksi = model_multikelas(x)

print("Hasil prediksi (Logits):", prediksi.detach())
print("Shape prediksi:", prediksi.shape) 
# Harus (1, 3) karena 1 baris data menghasilkan 3 tebakan kelas.

## Fase 3: Menangani Data Dummy

Kita buat dataset sintetis yang lebih besar menggunakan `make_classification` dari Scikit-Learn agar kita bisa melihat proses training.

In [ ]:
from sklearn.datasets import make_classification
import torch.utils.data as data

X_np, y_np = make_classification(n_samples=1000, n_features=10, n_classes=3, n_informative=5, random_state=42)

# Ubah ke bentuk Tensor PyTorch
X_tensor = torch.tensor(X_np, dtype=torch.float32)
y_tensor = torch.tensor(y_np, dtype=torch.long)  # kelas harus integer (long)

dataset = data.TensorDataset(X_tensor, y_tensor)
dataloader = data.DataLoader(dataset, batch_size=16, shuffle=True)

print("Total batch:", len(dataloader))
print("Shape X per batch:", next(iter(dataloader))[0].shape)

## Fase 4: Lima Baris Inti (Loop Training)

Sekarang kita tentukan Loss dan Optimizer.
Karena ini tugas **multiclass**, kita wajib menggunakan `CrossEntropyLoss()`.

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_multikelas.parameters(), lr=0.01)

epochs = 10

for epoch in range(epochs):
    total_loss = 0.0
    for batch_X, batch_y in dataloader:
        # 5 langkah dasar PyTorch
        optimizer.zero_grad()                 # 1. Reset gradient lama
        logits = model_multikelas(batch_X)    # 2. Forward pass
        loss = criterion(logits, batch_y)     # 3. Hitung loss
        loss.backward()                       # 4. Hitung gradient tiap parameter
        optimizer.step()                      # 5. Update parameter

        total_loss += loss.item()

    rata_rata_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch+1}/{epochs} | Loss: {rata_rata_loss:.4f}")

Nilai loss turun seiring berjalannya epoch.

---

## Fase 5: Eksperimen Ketidakcocokan (Praktik Mandiri)

Sebagai asisten riset, kita harus tahu apa yang terjadi jika terjadi ketidakcocokan konfigurasi.

**Tugas:**
Di cell bawah ini, salin kode pembuatan model dan loop training di atas, lalu terapkan modifikasi berikut:
1. Ubah arsitekturnya untuk mengeluarkan **1 angka saja** (untuk tugas regresi: `nn.Linear(64, 1)`)
2. Biarkan target datanya tetap integer multikelas (`batch_y`)
3. Gunakan loss MSE: `criterion = nn.MSELoss()`
4. MSELoss butuh tipe data *float*, ubah tipe target dengan `batch_y.float()`.

Jalankan kodenya, dan tuliskan observasi mengenai proses training tersebut. Apakah terjadi error? Apakah loss turun secara wajar?

In [ ]:
# TULIS KODE MISMATCH ANDA DI SINI
# 1. Buat model_mismatch
# 2. Tentukan MSELoss
# 3. Lakukan Loop 5 baris sakti



**Observasi Mismatch:**

*(Tulis jawaban Anda di sini: apa yang aneh dari training mismatch di atas?)*

---
**Selesai!** Anda telah menulis forward pass, merakit model, dan menjalankan 5 langkah dasar PyTorch secara eksplisit.